In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador" 

In [0]:
#Cria o Widget para seleção do mês por extenso
meses_do_ano = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

dbutils.widgets.dropdown(
    name="Mes_Extenso", 
    defaultValue="JANEIRO", 
    choices=meses_do_ano, 
    label="Selecione o Mês por Extenso"
)

Meses = dbutils.widgets.get("Mes_Extenso")

In [0]:
caminho_arquivos_individual = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Calculista/TRABALHISTA_MASSA/FATURAMENTO CALCULISTA {Meses} TRABALHISTA.xlsx'
caminho_arquivos_coletivo = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Calculista/TRABALHISTA_COLETIVO/FATURAMENTO CALCULISTA {Meses} COLETIVO.xlsx'
caminho_arquivo_precificacao = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Calculista/PRECIFICACAO/Tabela de Precificação Calculistas e Peritos - Versão Atualizada 3.xlsx'

df_faturamento_individual =   pd.read_excel(
            caminho_arquivos_individual,
            sheet_name="TRAB - HISTÓRICO DE CÁLCULOS",
            header=2,
            engine="openpyxl")

df_faturamento_coletivo =   pd.read_excel(
            caminho_arquivos_coletivo,
            sheet_name="TRAB - HISTÓRICO DE CÁLCULOS",
            header=0,
            engine="openpyxl") 
            
df_precificacao =   pd.read_excel(
            caminho_arquivo_precificacao,
            sheet_name="Precificação Cálculos",
            header=2,
            engine="openpyxl")

df_desconto_sla_precificacao = pd.read_excel(
            caminho_arquivo_precificacao,
            sheet_name="SLA - PENALIDADE - CÁLCULOS",
            header=16,
            engine="openpyxl")
            
df_sla_precificacao = pd.read_excel(
            caminho_arquivo_precificacao,
            sheet_name="SLA - PENALIDADE - CÁLCULOS",
            header=1,
            engine="openpyxl")

In [0]:
df = spark.read.table("databox.juridico_comum.br_consolidado_calculos")

df_faturamento_cosolidado_historico = df.toPandas()

**CONCATENANDO OS DOIS DF:**

In [0]:
df_concatenado = df_faturamento_individual
df_concatenado = pd.concat([df_faturamento_individual, df_faturamento_coletivo], ignore_index=True) #<===== USAR QUANDO TIVER FATURAMENTOS COLETIVOS 

**AJUSTE DA COLUNA OBSERVAÇÃO E REMOÇÃO DE TEXTO DA TAREFA DO ELAW**

In [0]:
df_concatenado['OBSERVAÇÃO'] = df_concatenado['OBSERVAÇÃO'].str.replace('CONTADOR: EFETUAR O CÁLCULO - ', '', regex=False)
df_concatenado = df_concatenado.rename(columns={'OBSERVAÇÃO': 'NATUREZA DO SERVIÇO'})

**AJUSTA DA COLUNA ANÁLISE VIA E CONDICIONAL PARA A COLUNA "FATURAR ?"**

In [0]:
df_concatenado = df_concatenado.rename(columns={'ANÁLISE VIA': 'ANÁLISE GCB'})

In [0]:
df_concatenado.loc[df_concatenado['ANÁLISE GCB'].str.contains('FATURAR', case=False, na=False), 'FATURAR?'] = 'FATURADO'

In [0]:
df_concatenado.loc[df_concatenado['ANÁLISE GCB'].str.contains('NÃO FATURAR', case=False, na=False), 'FATURAR?'] = 'NÃO FATURADO'

**AJUSTADO TABELA DE COMISSÃO DOS ESCRITÓRIOS PARA FORMATO LONGO**

In [0]:
df_precificacao_long = df_precificacao.melt(
    id_vars='TIPO DE TAREFA',
    var_name='ESCRITÓRIOS',
    value_name='VALOR_TAREFA'
).dropna()

**MERGE DO DF DE COMISSÃO COM A DF_CONCATENADO PARA OS CASOS A FATURAR**

In [0]:
df_resultado = df_concatenado.merge(
     df_precificacao_long,
     left_on=['ESCRITÓRIO CONTADOR*', 'NATUREZA DO SERVIÇO'],
     right_on=['ESCRITÓRIOS', 'TIPO DE TAREFA' ],
     how='left'
 )

In [0]:
df_resultado.loc[df_resultado['FATURAR?'] == 'NÃO FATURADO', 'VALOR_TAREFA'] = 0

In [0]:
df_resultado = df_resultado.drop(columns=['ESCRITÓRIOS'])
df_resultado = df_resultado.drop(columns=['TIPO DE TAREFA'])

**COLUNA PARA VERIFICAR O DESCONTO EM CASO DE DESCUMPRIMENTO DO SLA DO ESCRITÓRIO**

In [0]:
df_desconto_sla_precificao_long = df_desconto_sla_precificacao.melt(
    id_vars='TIPO DE TAREFA',
    var_name='ESCRITÓRIO',    
    value_name='DESCONTO SLA'
).dropna()

In [0]:
df_final = df_resultado.merge(
    df_desconto_sla_precificao_long,
    left_on=['NATUREZA DO SERVIÇO', 'ESCRITÓRIO CONTADOR*'],
    right_on=['TIPO DE TAREFA', 'ESCRITÓRIO'],
    how='left'
)

In [0]:
def aplicar_penalidade(row):
    # Só aplica penalidade se a tarefa estiver confirmada com atraso
    if row['STATUS DA TAREFA'] != 'Confirmados em Atraso':
        return row['VALOR_TAREFA']
    
    # Não aplica penalidade se o valor a ser pago for nulo ou zero
    if row['VALOR_TAREFA'] <= 0 or pd.isna(row['DESCONTO SLA']):
        return row['VALOR_TAREFA']
    
    # Não há penalidade
    if pd.isna(row['DESCONTO SLA']):
        return row['VALOR_TAREFA']
    
    # Penalidade percentual
    if row['DESCONTO SLA'] < 1  :
        return row['VALOR_TAREFA'] * (1 - row['DESCONTO SLA'])
    
    # Penalidade em valor absoluto
    return row['VALOR_TAREFA'] - row['DESCONTO SLA']

In [0]:
df_final['VALOR_FINAL'] = df_final.apply(aplicar_penalidade, axis=1)

In [0]:
df_final.loc[df_final['STATUS DA TAREFA'] == 'Confirmados', 'DESCONTO SLA'] = 0

In [0]:
df_final.drop(columns=['TIPO DE TAREFA'], inplace=True)
df_final.drop(columns=['ESCRITÓRIO'], inplace=True)

**VALIDAÇÃO PAGAMENTO SE JÁ HOUVE PAGAMENTO NO MÊS**

In [0]:
df_final['DATA DE CONFIRMAÇÃO'] = pd.to_datetime(df_final['DATA DE CONFIRMAÇÃO'], dayfirst=True)

In [0]:
df_final['CHAVE_VALIDACAO'] = df_final['PROCESSO - ID'].astype(str) + '_' + df_final['NATUREZA DO SERVIÇO'].astype(str) + '_' + df_final['DATA DE CONFIRMAÇÃO'].astype(str)

In [0]:
df_faturamento_cosolidado_historico.head()

In [0]:
df_final['STATUS_PAGAMENTO'] = df_final['CHAVE_VALIDACAO'].isin(df_faturamento_cosolidado_historico['ID_NATUREZA_DATA_CONFIRMACAO'])

# Substitui True/False por textos desejados
df_final['STATUS_PAGAMENTO'] = df_final['STATUS_PAGAMENTO'].map({
    True: 'Pago no mês',
    False: 'Não pago no mês'
})


**ZERA O PAGAMENTO CASO JÁ TENHA UMA CHAVE COM AQUELE VALOR PAGO NO MÊS**

In [0]:
df_final.loc[df_final['STATUS_PAGAMENTO'] == 'Pago no mês', 'VALOR_FINAL'] = 0
df_final.loc[df_final['STATUS_PAGAMENTO'] == 'Pago no mês', 'DESCONTO SLA'] = 0

In [0]:
df_final.drop(columns=['CHAVE_VALIDACAO'], inplace=True)

**QUEBRA DE DF POR ESCRITÓRIO**

In [0]:
df_final = df_final.rename(columns={'ESCRITÓRIO CONTADOR*': 'ESCRITÓRIO CONTADOR'})

In [0]:
DIRETORIO_SAIDA = '/Volumes/databox/juridico_comum/arquivos/bases_pagamentos/'

# Coluna que será usada para dividir
coluna_base = 'ESCRITÓRIO CONTADOR'

# Pega os valores únicos (ignorando nulos)
valores_unicos = df_final[coluna_base].dropna().unique()

print(f"Iniciando exportação para {len(valores_unicos)} escritórios...")

for valor in valores_unicos:
    # Filtra o DataFrame
    df_filtrado = df_final[df_final[coluna_base] == valor]

    # Limpeza do nome do arquivo para evitar erros de caminho
    valor_str = str(valor).strip().replace(' ', '_').replace('/', '-').replace('\\', '-')
    nome_arquivo = f"{coluna_base}_{valor_str}.xlsx"
    
    caminho_local_temporario = f"/tmp/{nome_arquivo}"
    caminho_final_destino = f"/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Calculista/FATURAMENTO_FINAL/{nome_arquivo}"

    try:
        # 1. Salva no /tmp (local rápido e seguro)
        df_filtrado.to_excel(caminho_local_temporario, index=False)

        # 2. Copia APENAS O CONTEÚDO para o destino final (evita o erro de Permissão/utime)
        shutil.copyfile(caminho_local_temporario, caminho_final_destino)
        
        # 3. Remove o arquivo temporário
        os.remove(caminho_local_temporario)

        print(f"Arquivo salvo com sucesso: {nome_arquivo}")

    except Exception as e:
        print(f"ERRO ao processar {nome_arquivo}: {e}")